# Improved Report Recommendation Pipeline

### Key improvements over the original approach:

1. **Soft category boosting** — No hard 0.70 cutoff. All LLM-recommended categories are kept; their relevance scores become boost weights.
2. **Multi-query retrieval** — One focused search query per category (instead of one combined query), producing tighter embeddings.
3. **Weighted Reciprocal Rank Fusion (RRF)** — Merges vector + BM25 results across all per-category queries, weighted by category relevance.
4. **Cross-encoder reranking** — Final reranking of top candidates with a cross-encoder that sees (profile, report) pairs jointly.
5. **Expanded candidate pool** — More reports considered due to softer filtering, fewer missed gems.

In [1]:
import json
import os
import logging
from typing import List, Dict, Any
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
)
logger = logging.getLogger("improved_recommendation")

with open("linkedin_scrape_results_20260320_120417.json", "r") as f:
    profiles_data = json.load(f)

with open("catagories.json", "r") as f:
    categories = json.load(f)

profile = profiles_data["results"][0]
url = profile["profile_url"]
print(f"Profile: {profile.get('scraped_job_title')} at {profile.get('recent_company_name')}")
print(f"Industry: {profile.get('recent_company_industry')}")
print(f"URL: {url}")

Profile: Consultant at Pioneer Management Consulting
Industry: Business Consulting and Services
URL: http://www.linkedin.com/in/morgan-dammann-a48667140


In [2]:
def recommend_categories(
    profile: dict,
    categories: list[dict] | None = None,
) -> list[dict]:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    if categories is None:
        with open("catagories.json", "r") as f:
            categories = json.load(f)

    system_prompt = """You are an expert at matching professional profiles to relevant report categories.
You will receive a person's LinkedIn profile data and a list of category groups.
From ALL the groups, pick only the subcategories that are relevant to this person.
Return ONLY valid JSON."""

    user_prompt = f"""Analyze this LinkedIn profile and recommend the most relevant subcategories.

## Profile
{json.dumps(profile, indent=2)}

## Available Category Groups
{json.dumps(categories, indent=2)}

Return a JSON object with this exact structure:
{{
    "recommended": [
        {{"category": "<subcategory name>", "relevance_score": <float 0-1>, "reasoning": "<one sentence>"}},
        {{"category": "<subcategory name>", "relevance_score": <float 0-1>, "reasoning": "<one sentence>"}}
    ]
}}

Rules:
- Only pick subcategories from the provided lists (e.g. "Management", "Energy", "Global", "Deep Tech").
- Only include subcategories that are genuinely relevant to this person's profile.
- Return them ranked by relevance_score (highest first)."""

    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"},
        temperature=0.3,
    )

    return json.loads(response.choices[0].message.content)["recommended"]

In [3]:
all_recommended = recommend_categories(profile, categories)

print("All LLM-recommended categories (NO hard cutoff applied):\n")
for cat in all_recommended:
    print(f"  {cat['category']:30s}  score={cat['relevance_score']:.2f}  | {cat['reasoning']}")

2026-03-24 13:40:40,129 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


All LLM-recommended categories (NO hard cutoff applied):

  management                      score=0.97  | They are currently a consultant at a management consulting firm specializing in business strategy, organizational change, program management, and transformation.
  employment                      score=0.72  | Their experience includes customer success, organizational design exposure, HR-related role definition work, and consulting in workforce and operating model contexts tied to employment topics.
  work-of-future                  score=0.68  | The profile shows involvement in organizational change, business transformation, analytics, and redesign of job roles, which aligns with future-of-work themes.
  industrials                     score=0.41  | An internship as a District Manager Intern at ALDI and consulting exposure to operational improvement provide some relevance to industrials-related business operations.
  america                         score=0.39  | Their education an

## Improvement #1: Soft Category Filtering

Instead of a hard cutoff at 0.70 (which dropped `work-of-future` at 0.68 and `america` at 0.61),
we keep **all** categories above a very low threshold (0.30) and carry their relevance scores
forward as **boost weights**. This expands the candidate pool and prevents losing relevant reports.

In [4]:
SOFT_THRESHOLD = 0.30

kept_categories = [c for c in all_recommended if c["relevance_score"] >= SOFT_THRESHOLD]
category_weights = {c["category"]: c["relevance_score"] for c in kept_categories}
category_slugs = list(category_weights.keys())

print(f"Kept {len(kept_categories)}/{len(all_recommended)} categories (threshold >= {SOFT_THRESHOLD}):\n")
for slug, weight in category_weights.items():
    bar = '█' * int(weight * 20)
    print(f"  {slug:30s}  weight={weight:.2f}  {bar}")

Kept 5/5 categories (threshold >= 0.3):

  management                      weight=0.97  ███████████████████
  employment                      weight=0.72  ██████████████
  work-of-future                  weight=0.68  █████████████
  industrials                     weight=0.41  ████████
  america                         weight=0.39  ███████


In [5]:
DATA_FILE = "test_data.json"
ALLOWED_CATEGORY_SLUGS = {"for-you", "sector", "theme", "region"}

with open(DATA_FILE, "r") as f:
    db_data = json.load(f)

db_categories = db_data["categories"]
db_subcategories = db_data["subcategories"]
db_reports = db_data["reports"]
db_report_subcategory = db_data["report_subcategory"]

subcat_by_slug = {sc["slug"]: sc for sc in db_subcategories}
report_by_id = {r["id"]: r for r in db_reports}
allowed_cat_ids = {
    cat["id"] for cat in db_categories if cat["slug"] in ALLOWED_CATEGORY_SLUGS
}

valid_slugs = [
    s for s in category_slugs
    if s in subcat_by_slug and subcat_by_slug[s]["cat_id"] in allowed_cat_ids
]
slug_to_subcat_id = {s: subcat_by_slug[s]["id"] for s in valid_slugs}
subcat_ids = set(slug_to_subcat_id.values())

report_id_set = sorted({
    link["report_id"]
    for link in db_report_subcategory
    if link["subcat_id"] in subcat_ids
})

reports = [
    {"id": rid, "slug": report_by_id[rid]["slug"]}
    for rid in report_id_set
    if rid in report_by_id
]
report_ids = [r["id"] for r in reports]
report_slug_by_id = {r["id"]: r["slug"] for r in reports}

# Reverse mapping: report_id -> set of matching user-relevant category slugs
report_to_slugs: Dict[int, set] = {}
for link in db_report_subcategory:
    if link["subcat_id"] in subcat_ids:
        rid = link["report_id"]
        if rid not in report_to_slugs:
            report_to_slugs[rid] = set()
        for slug, sid in slug_to_subcat_id.items():
            if link["subcat_id"] == sid:
                report_to_slugs[rid].add(slug)

print(f"Valid slugs resolved: {valid_slugs}")
print(f"Expanded candidate pool: {len(reports)} reports")
print(f"Reports loaded from DB: {len(db_reports)}")


Valid slugs resolved: ['management', 'employment', 'work-of-future', 'industrials', 'america']
Expanded candidate pool: 456 reports
Reports loaded from DB: 672


In [6]:
report_to_slugs

{1: {'management'},
 2: {'management', 'work-of-future'},
 3: {'employment', 'management'},
 4: {'industrials', 'management'},
 5: {'industrials', 'management'},
 6: {'employment', 'management'},
 7: {'management', 'work-of-future'},
 8: {'management', 'work-of-future'},
 9: {'management'},
 10: {'management', 'work-of-future'},
 11: {'employment', 'management'},
 12: {'management', 'work-of-future'},
 13: {'employment', 'management', 'work-of-future'},
 14: {'management', 'work-of-future'},
 15: {'america', 'industrials', 'management'},
 16: {'industrials', 'management'},
 17: {'industrials', 'management'},
 18: {'industrials', 'management'},
 19: {'industrials', 'management'},
 20: {'management'},
 21: {'management'},
 22: {'management'},
 23: {'management'},
 24: {'management', 'work-of-future'},
 25: {'management', 'work-of-future'},
 26: {'management', 'work-of-future'},
 27: {'management', 'work-of-future'},
 28: {'management', 'work-of-future'},
 29: {'management', 'work-of-futu

In [7]:
import boto3
from botocore.config import Config
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import OpenSearchVectorSearch
from requests_aws4auth import AWS4Auth
from opensearchpy import OpenSearch, RequestsHttpConnection
from urllib.parse import urlparse

HUGGINGFACE_EMBEDDINGS_MODEL = os.getenv("HUGGINGFACE_EMBEDDINGS_MODEL")
EMBEDDING_MODEL = HuggingFaceEmbeddings(
    model_name=HUGGINGFACE_EMBEDDINGS_MODEL, show_progress=False
)

AWS_OPENSEARCH_ACCESS_KEY_ID = os.getenv("AWS_OPENSEARCH_ACCESS_KEY_ID")
AWS_OPENSEARCH_SECRET_ACCESS_KEY = os.getenv("AWS_OPENSEARCH_SECRET_ACCESS_KEY")
AWS_OPENSEARCH_REGION_NAME = os.getenv("AWS_OPENSEARCH_REGION_NAME")
OPENSEARCH_URL = os.getenv("OPENSEARCH_URL")
OPENSEARCH_INDEX = "ghost_research_report_overviews"

credentials = boto3.Session(
    aws_access_key_id=AWS_OPENSEARCH_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_OPENSEARCH_SECRET_ACCESS_KEY,
).get_credentials()

awsauth = AWS4Auth(
    AWS_OPENSEARCH_ACCESS_KEY_ID,
    AWS_OPENSEARCH_SECRET_ACCESS_KEY,
    AWS_OPENSEARCH_REGION_NAME,
    "es",
    session_token=credentials.token,
)

VECTOR_DB = OpenSearchVectorSearch(
    embedding_function=EMBEDDING_MODEL,
    opensearch_url=OPENSEARCH_URL,
    http_auth=awsauth,
    timeout=300,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    index_name=OPENSEARCH_INDEX,
    engine="faiss",
)

parsed_url = urlparse(OPENSEARCH_URL)
os_client = OpenSearch(
    hosts=[{"host": parsed_url.hostname, "port": parsed_url.port or 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300,
)

/home/dixit/Desktop/report-recommendation/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-24 13:40:46,203 [INFO] sentence_transformers.SentenceTransformer - Use pytorch device_name: cpu
2026-03-24 13:40:46,204 [INFO] sentence_transformers.SentenceTransformer - Load pretrained SentenceTransformer: sentence-transformers/LaBSE
2026-03-24 13:40:51,585 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/sentence-transformers/LaBSE/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-24 13:40:51,616 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/LaBSE/836121a0533e5664b21c7aacc5d22951f2b8b25b/modules.json "HTTP/1.1 200 OK"
2026-03-24 13:40:51,902 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/sentence-transforme

## Improvement #2: Multi-Query Retrieval

Instead of generating **one combined search query** that tries to cover all categories (producing a diluted embedding),
we generate **one focused query per category**. Each query produces a tighter embedding that lands closer to the
relevant report cluster in the vector space.

In [8]:
def generate_focused_query(profile: dict, category: dict) -> str:
    """Generate a focused 1-2 sentence search query for a single category."""
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    prompt = f"""Based on this LinkedIn profile, write a 1-2 sentence search query
to find research reports relevant to the \"{category['category']}\" topic area.

Focus on what specifically about {category['category']} matters to this person
given their role, industry, and experience. Be concrete and specific.

Profile:
{json.dumps(profile, indent=2)}

Category context: {category['reasoning']}

Write ONLY the 1-2 sentence search query, nothing else."""

    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()

In [9]:
category_queries = {}
for cat in kept_categories:
    query = generate_focused_query(profile, cat)
    category_queries[cat["category"]] = query
    print(f"[{cat['category']}] (weight={cat['relevance_score']:.2f}):")
    print(f"  {query}\n")

print(f"Generated {len(category_queries)} focused queries (vs 1 in old approach)")

2026-03-24 13:41:02,118 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[management] (weight=0.97):
  Find research reports on management consulting best practices for organizational change, business transformation, and program management, especially in mid-sized companies. Prioritize reports on change management execution, organizational design/effectiveness, stakeholder alignment, and managing cross-functional transformation initiatives to improve client outcomes.



2026-03-24 13:41:05,420 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[employment] (weight=0.72):
  Find research reports on employment trends in management consulting and business transformation, especially talent demand for consultants with customer success, organizational design, workforce planning, and operating model/change management experience in mid-sized firms. Also look for reports on job architecture, role redesign, internal mobility, and early-career talent development programs in professional services and corporate strategy/HR functions.



2026-03-24 13:41:07,878 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[work-of-future] (weight=0.68):
  Find research reports on the future of work in management consulting and business transformation, focused on how AI, automation, and advanced analytics are reshaping consultant, customer success, and business application roles, including implications for organizational design, role redesign, and workforce planning. Prioritize reports on change management, skills-based workforce models, engineering/leadership rotation programs, and talent development strategies for mid-sized professional services firms.



2026-03-24 13:41:11,257 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[industrials] (weight=0.41):
  Research reports on industrials sector operational improvement, with a focus on supply chain optimization, workforce/organizational design, cost transformation, and business process efficiency for distributors, manufacturers, and logistics-heavy operators. Prioritize reports covering consulting-relevant themes such as change management, program management, advanced analytics, and margin improvement in industrial and retail-adjacent operations.



2026-03-24 13:41:14,027 [INFO] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[america] (weight=0.39):
  Find research reports on U.S. business transformation and management consulting trends, especially in the Midwest/Minneapolis market, covering organizational change, operating model redesign, program management, data/advanced analytics, and strategic cost optimization. Prioritize reports on how American companies are investing in consulting-led transformation, organizational effectiveness, and M&A integration across industries such as retail, enterprise software, and business services.

Generated 5 focused queries (vs 1 in old approach)


In [10]:
category_queries

{'management': 'Find research reports on management consulting best practices for organizational change, business transformation, and program management, especially in mid-sized companies. Prioritize reports on change management execution, organizational design/effectiveness, stakeholder alignment, and managing cross-functional transformation initiatives to improve client outcomes.',
 'employment': 'Find research reports on employment trends in management consulting and business transformation, especially talent demand for consultants with customer success, organizational design, workforce planning, and operating model/change management experience in mid-sized firms. Also look for reports on job architecture, role redesign, internal mobility, and early-career talent development programs in professional services and corporate strategy/HR functions.',
 'work-of-future': 'Find research reports on the future of work in management consulting and business transformation, focused on how AI, a

## Improvement #3: Weighted RRF Across All Queries

For each category query, we run **both** vector search and BM25 keyword search.
All result lists are merged via **Reciprocal Rank Fusion**, weighted by category relevance score.

Formula: `RRF_score(report) = Σ (category_weight / (k + rank))` across all result lists.

Reports that rank well in high-weight categories and across both retrieval methods float to the top.

In [11]:
RRF_K = 60
RESULTS_PER_QUERY = 20

all_rankings = []  # list of (weight, ranking_dict)
content_lookup = {}

for slug, query_text in category_queries.items():
    weight = category_weights[slug]

    # Vector search for this category query
    vector_docs = VECTOR_DB.similarity_search_with_score(
        query=query_text,
        k=RESULTS_PER_QUERY,
        filter={"terms": {"metadata.report_id": report_ids}},
    )
    vector_ranking = {}
    for rank, (doc, score) in enumerate(vector_docs):
        rid = doc.metadata["report_id"]
        vector_ranking[rid] = rank
        content_lookup[rid] = doc.page_content

    # BM25 keyword search for this category query
    bm25_body = {
        "size": RESULTS_PER_QUERY,
        "query": {
            "bool": {
                "must": [{"match": {"text": query_text}}],
                "filter": [{"terms": {"metadata.report_id": report_ids}}],
            }
        },
    }
    bm25_response = os_client.search(index=OPENSEARCH_INDEX, body=bm25_body)
    bm25_ranking = {}
    for rank, hit in enumerate(bm25_response["hits"]["hits"]):
        rid = hit["_source"]["metadata"]["report_id"]
        bm25_ranking[rid] = rank
        if rid not in content_lookup:
            content_lookup[rid] = hit["_source"].get("text", "")

    all_rankings.append((weight, vector_ranking))
    all_rankings.append((weight, bm25_ranking))

    print(f"[{slug}] Vector: {len(vector_ranking)} | BM25: {len(bm25_ranking)}")

# Weighted RRF across ALL result lists
all_candidate_ids = set()
for _, ranking in all_rankings:
    all_candidate_ids |= set(ranking.keys())

rrf_scores = {}
for rid in all_candidate_ids:
    score = 0.0
    for weight, ranking in all_rankings:
        if rid in ranking:
            score += weight / (RRF_K + ranking[rid])
    rrf_scores[rid] = score

rrf_ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

print(f"\nTotal unique candidates from multi-query RRF: {len(rrf_ranked)}")
print(f"\nTop 10 by RRF score (before cross-encoder reranking):")
print("=" * 80)
for i, (rid, score) in enumerate(rrf_ranked[:10], 1):
    cats = report_to_slugs.get(rid, set())
    slug = report_slug_by_id.get(rid, "unknown")
    print(f"{i:2d}. Report {rid:>4d}  RRF={score:.6f}  cats={cats}")
    print(f"    slug: {slug}")
    print(f"    {content_lookup.get(rid, 'N/A')[:150]}...\n")

2026-03-24 13:41:26,570 [INFO] opensearch - POST https://search-consultai-production-zs6diszzmes4wblq2ix5oq5tea.eu-central-1.es.amazonaws.com:443/ghost_research_report_overviews/_search [status:200 request:12.378s]
2026-03-24 13:41:28,086 [INFO] opensearch - POST https://search-consultai-production-zs6diszzmes4wblq2ix5oq5tea.eu-central-1.es.amazonaws.com:443/ghost_research_report_overviews/_search [status:200 request:1.509s]


[management] Vector: 20 | BM25: 20


2026-03-24 13:41:28,930 [INFO] opensearch - POST https://search-consultai-production-zs6diszzmes4wblq2ix5oq5tea.eu-central-1.es.amazonaws.com:443/ghost_research_report_overviews/_search [status:200 request:0.695s]
2026-03-24 13:41:29,366 [INFO] opensearch - POST https://search-consultai-production-zs6diszzmes4wblq2ix5oq5tea.eu-central-1.es.amazonaws.com:443/ghost_research_report_overviews/_search [status:200 request:0.429s]


[employment] Vector: 20 | BM25: 20


2026-03-24 13:41:30,103 [INFO] opensearch - POST https://search-consultai-production-zs6diszzmes4wblq2ix5oq5tea.eu-central-1.es.amazonaws.com:443/ghost_research_report_overviews/_search [status:200 request:0.574s]
2026-03-24 13:41:30,386 [INFO] opensearch - POST https://search-consultai-production-zs6diszzmes4wblq2ix5oq5tea.eu-central-1.es.amazonaws.com:443/ghost_research_report_overviews/_search [status:200 request:0.276s]


[work-of-future] Vector: 20 | BM25: 20


2026-03-24 13:41:31,043 [INFO] opensearch - POST https://search-consultai-production-zs6diszzmes4wblq2ix5oq5tea.eu-central-1.es.amazonaws.com:443/ghost_research_report_overviews/_search [status:200 request:0.542s]
2026-03-24 13:41:31,316 [INFO] opensearch - POST https://search-consultai-production-zs6diszzmes4wblq2ix5oq5tea.eu-central-1.es.amazonaws.com:443/ghost_research_report_overviews/_search [status:200 request:0.267s]


[industrials] Vector: 20 | BM25: 20


2026-03-24 13:41:32,017 [INFO] opensearch - POST https://search-consultai-production-zs6diszzmes4wblq2ix5oq5tea.eu-central-1.es.amazonaws.com:443/ghost_research_report_overviews/_search [status:200 request:0.539s]
2026-03-24 13:41:32,344 [INFO] opensearch - POST https://search-consultai-production-zs6diszzmes4wblq2ix5oq5tea.eu-central-1.es.amazonaws.com:443/ghost_research_report_overviews/_search [status:200 request:0.320s]


[america] Vector: 20 | BM25: 20

Total unique candidates from multi-query RRF: 92

Top 10 by RRF score (before cross-encoder reranking):
 1. Report  154  RRF=0.087989  cats={'industrials', 'work-of-future', 'management'}
    slug: professional-services-2030-remote-work-ai-consulting-and-skills-evolution
    The report explores the transformation of the professional services sector driven by remote work, AI consulting, and the evolution of skills up to 203...

 2. Report  481  RRF=0.077771  cats={'management'}
    slug: ai-driven-risk-analytics-workforce-automation
    This report provides strategic insights into the impact of AI-driven risk analytics on workforce automation within the insurance and banking sectors. ...

 3. Report  323  RRF=0.075529  cats={'work-of-future', 'management'}
    slug: transforming-consulting-in-ai-era
    This report delves into how AI and digital disruptions are reshaping the consulting industry. It explores changes in business models, service delivery...

## Improvement #4: Cross-Encoder Reranking

The cross-encoder sees both the **profile description** and the **report description** together,
scoring their relevance jointly. This catches nuances that independent embeddings miss.

We take the top 30 candidates from RRF and rerank them. This is the industry standard
two-stage retrieval pattern (fast bi-encoder retrieval -> accurate cross-encoder reranking).

In [12]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")
print("Cross-encoder loaded.")

2026-03-24 13:41:32,764 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-12-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-24 13:41:33,067 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L12-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-24 13:41:33,103 [INFO] httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L12-v2/7b0235231ca2674cb8ca8f022859a6eba2b1c968/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5209.83it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-24 

Cross-encoder loaded.


In [13]:
TOP_N_FOR_RERANK = 30

# Build a concise profile summary for the cross-encoder query side
profile_parts = []
if profile.get("scraped_job_title"):
    profile_parts.append(f"{profile['scraped_job_title']} at {profile.get('recent_company_name', '')}")
if profile.get("recent_company_industry"):
    profile_parts.append(f"Industry: {profile['recent_company_industry']}")
if profile.get("recent_company_details", {}).get("specialties"):
    profile_parts.append(f"Specialties: {profile['recent_company_details']['specialties']}")

exp_titles = []
for exp in profile.get("scraped_experience", [])[:6]:
    title = exp.get("title", "")
    company = exp.get("company_name", "")
    if title and not title.startswith("-") and company:
        exp_titles.append(f"{title} at {company}")
if exp_titles:
    profile_parts.append(f"Experience: {'; '.join(exp_titles)}")

profile_summary = ". ".join(profile_parts)
print(f"Profile summary for reranking ({len(profile_summary)} chars):\n")
print(profile_summary)

Profile summary for reranking (677 chars):

Consultant at Pioneer Management Consulting. Industry: Business Consulting and Services. Specialties: Management Consulting, Organizational Change, Data Analytics, Business Strategy, Organizational Design, Advanced Analytics, Program Management, M&A Integration, Strategic Cost Optimization, Organizational Restructuring, Organizational Effectiveness, Service Now, and Business Transformation. Experience: Consultant at Pioneer Management Consulting; Full-time · 4 yrs 11 mos at Customer Success Manager; Vice President at University of St. Thomas Enactus; Economics Tutor at University of St. Thomas; District Manager Intern at ALDI USA; Vice President of Relations at Enactus


In [14]:
candidates = rrf_ranked[:TOP_N_FOR_RERANK]

pairs = []
candidate_rids = []
for rid, rrf_score in candidates:
    content = content_lookup.get(rid, "")
    pairs.append((profile_summary, content))
    candidate_rids.append(rid)

rerank_scores = reranker.predict(pairs)

final_results = []
for rid, (_, rrf_score), rerank_score in zip(candidate_rids, candidates, rerank_scores):
    final_results.append({
        "report_id": rid,
        "report_slug": report_slug_by_id.get(rid, "unknown"),
        "content": content_lookup.get(rid, ""),
        "rrf_score": rrf_score,
        "rerank_score": float(rerank_score),
        "matching_categories": sorted(report_to_slugs.get(rid, set())),
    })

final_results.sort(key=lambda x: x["rerank_score"], reverse=True)

print(f"{'=' * 90}")
print(f"TOP 10 RECOMMENDED REPORTS (Cross-Encoder Reranked)")
print(f"{'=' * 90}\n")

for i, rec in enumerate(final_results[:10], 1):
    print(f"{i:2d}. Report ID: {rec['report_id']}")
    print(f"    Slug: {rec['report_slug']}")
    print(f"    Rerank Score: {rec['rerank_score']:.4f}  |  RRF Score: {rec['rrf_score']:.6f}")
    print(f"    Categories: {rec['matching_categories']}")
    print(f"    {rec['content'][:200]}...")
    print()

Batches: 100%|██████████| 1/1 [00:02<00:00,  2.77s/it]

TOP 10 RECOMMENDED REPORTS (Cross-Encoder Reranked)

 1. Report ID: 323
    Slug: transforming-consulting-in-ai-era
    Rerank Score: 6.2634  |  RRF Score: 0.075529
    Categories: ['management', 'work-of-future']
    This report delves into how AI and digital disruptions are reshaping the consulting industry. It explores changes in business models, service delivery, and client expectations, driven by AI, cloud com...

 2. Report ID: 134
    Slug: human-capital-trends-in-professional-services-outsourcing-remote-work-and-ai
    Rerank Score: 5.8962  |  RRF Score: 0.035856
    Categories: ['industrials', 'management', 'work-of-future']
    This report explores the transformative trends in the professional services industry focusing on South Asia and ASEAN. It delves into the integration of outsourcing, remote work, and artificial intell...

 3. Report ID: 139
    Slug: insurance-workforce-of-the-future-digital-skills-and-climate-risk-expertise
    Rerank Score: 5.8192  |  RRF Score: 0.04

In [ ]:
print("Final recommended reports (ordered by cross-encoder relevance):\n")
final_reports = [
    {"id": r["report_id"], "slug": r["report_slug"]}
    for r in final_results[:10]
]
for r in final_reports:
    print(f"  ID: {r['id']:>4d}  |  {r['slug']}")
print(f"\nReport IDs: {[r['id'] for r in final_reports]}")

Final recommended reports (ordered by cross-encoder relevance):

  ID:  323  |  transforming-consulting-in-ai-era
  ID:  134  |  human-capital-trends-in-professional-services-outsourcing-remote-work-and-ai
  ID:  139  |  insurance-workforce-of-the-future-digital-skills-and-climate-risk-expertise
  ID:   49  |  green-consulting-esg-driven-business-models-in-professional-services
  ID:  166  |  the-future-of-work-in-pharma-biotech
  ID:  141  |  human-capital-in-financial-services-shifts-toward-esg-and-digital-roles
  ID:  162  |  grocery-staples-workforce-transformation
  ID:  154  |  professional-services-2030-remote-work-ai-consulting-and-skills-evolution
  ID:  343  |  hybrid-office-models-global-trends-and-forecasts
  ID:  137  |  ai-automation-and-the-future-of-human-capital-in-software-it-services

Report IDs: [323, 134, 139, 49, 166, 141, 162, 154, 343, 137]


In [18]:

def get_report_url(slug: str) -> str:
    return f"https://www.ghostresearch.com/reports/{slug}"

report_url = []
for r in final_results[:10]:
    report_url.append(get_report_url(r["report_slug"]))

In [19]:
report_url

['https://www.ghostresearch.com/reports/transforming-consulting-in-ai-era',
 'https://www.ghostresearch.com/reports/human-capital-trends-in-professional-services-outsourcing-remote-work-and-ai',
 'https://www.ghostresearch.com/reports/insurance-workforce-of-the-future-digital-skills-and-climate-risk-expertise',
 'https://www.ghostresearch.com/reports/green-consulting-esg-driven-business-models-in-professional-services',
 'https://www.ghostresearch.com/reports/the-future-of-work-in-pharma-biotech',
 'https://www.ghostresearch.com/reports/human-capital-in-financial-services-shifts-toward-esg-and-digital-roles',
 'https://www.ghostresearch.com/reports/grocery-staples-workforce-transformation',
 'https://www.ghostresearch.com/reports/professional-services-2030-remote-work-ai-consulting-and-skills-evolution',
 'https://www.ghostresearch.com/reports/hybrid-office-models-global-trends-and-forecasts',
 'https://www.ghostresearch.com/reports/ai-automation-and-the-future-of-human-capital-in-soft

In [4]:
from recommend import get_report_recommendations_from_url
import json


urls = get_report_recommendations_from_url("https://www.linkedin.com/in/sayan-ghosh-a9781915b/")

2026-03-25 13:01:08,986 [INFO] Scraping LinkedIn profile: https://www.linkedin.com/in/sayan-ghosh-a9781915b/
2026-03-25 13:01:08,987 [INFO] Voyager API session initialized
2026-03-25 13:01:08,988 [INFO] Scraping profile: sayan-ghosh-a9781915b


TooManyRedirects: Exceeded 30 redirects.

In [ ]:
urls

['https://www.ghostresearch.com/reports/urban-w2e-projects-workforce-innovation',
 'https://www.ghostresearch.com/reports/geopolitical-decoupling-and-the-asia-first-doctrine',
 'https://www.ghostresearch.com/reports/deep-tech-materials-innovation-in-east-asia',
 'https://www.ghostresearch.com/reports/green-steel-aluminium-jobs-skills-supply-chain-transformation',
 'https://www.ghostresearch.com/reports/industrial-data-monetization-from-sensor-data-to-new-revenue-streams',
 'https://www.ghostresearch.com/reports/gcc-rare-earth-metals-supply-chain-disruptions',
 'https://www.ghostresearch.com/reports/us-defense-manufacturing-ramp-up-and-workforce-evolution',
 'https://www.ghostresearch.com/reports/trade-tariffs-and-real-estate-development-costs-a-global-analysis',
 'https://www.ghostresearch.com/reports/deep-tech-in-capital-goods-east-asia-focus',
 'https://www.ghostresearch.com/reports/sustaining-the-un-sustainables-investment-roadmap']

In [1]:
import json
from datetime import datetime
import os

li_at_token = os.getenv("LI_AT")

from linkedin_scraper import LinkedInScraper

scraper = LinkedInScraper(li_at_token=li_at_token)
result = scraper.scrape_profile("https://www.linkedin.com/in/dixit-vora-a140891a5/")
print(result)

2026-03-25 12:16:59,140 [INFO] Voyager API session initialized
2026-03-25 12:16:59,141 [INFO] Scraping profile: dixit-vora-a140891a5


2026-03-25 12:17:04,094 [INFO] Done — name='Dixit Vora' company='Jindal Saw Limited' skills=20 experience=6


{'profile_url': 'https://www.linkedin.com/in/dixit-vora-a140891a5/', 'status': 'success', 'scraped_name': 'Dixit Vora', 'scraped_headline': 'Technical Lead | Project Management & Industrial Design | Engineering Execution', 'scraped_about': '\u200bTechnical Lead | Project Management & Industrial Design Expert\n\u200bWith over 6 years of leadership experience, I specialize in the end-to-end execution of large-scale pipe manufacturing plant design, Installation and commissioning projects. I bridge the gap between complex engineering design and successful project delivery, managing budgets, technical teams, and multi-disciplinary layouts.\n\n\u200bMy expertise lies in designing and commissioning plants for LSAW, HSAW, and ERW pipe production, as well as advanced anti-corrosion coating systems (like 3LPE, 3LPP, FBE, CWC, CML). Includes AutoCAD drafting and SolidWorks modeling to the final installation of machines and ensure every machine and foundation is engineered for peak performance.', 

In [2]:
result

{'profile_url': 'https://www.linkedin.com/in/dixit-vora-a140891a5/',
 'status': 'success',
 'scraped_name': 'Dixit Vora',
 'scraped_headline': 'Technical Lead | Project Management & Industrial Design | Engineering Execution',
 'scraped_about': '\u200bTechnical Lead | Project Management & Industrial Design Expert\n\u200bWith over 6 years of leadership experience, I specialize in the end-to-end execution of large-scale pipe manufacturing plant design, Installation and commissioning projects. I bridge the gap between complex engineering design and successful project delivery, managing budgets, technical teams, and multi-disciplinary layouts.\n\n\u200bMy expertise lies in designing and commissioning plants for LSAW, HSAW, and ERW pipe production, as well as advanced anti-corrosion coating systems (like 3LPE, 3LPP, FBE, CWC, CML). Includes AutoCAD drafting and SolidWorks modeling to the final installation of machines and ensure every machine and foundation is engineered for peak performance